# Standalone RAGAS evaluation

This notebook contains the RAGAS evaluation pipeline formerly integrated into `backend/rag_pipeline.py`. It uses the existing persisted Chroma index and RAG answer pipeline, but runs independently of the FastAPI server and frontend. Run it from the project root after ingesting a PDF.

Update the two static paths in the next cell before running the notebook. The input workbook must contain question, reference-answer, and `Category` columns. Answers are generated for every row through the LangGraph chat workflow, but RAGAS metrics are calculated only for rows whose category is `Theoretical`; other rows are exported with blank metric cells.

In [1]:
from pathlib import Path

# STATIC PATHS: replace these values with your dataset and result locations.
INPUT_DATASET_PATH = Path(r"D:\AI_Bootcamp\genai-smart-education-platform\storage\uploads\RAG_Evaluation_Dataset.xlsx")
OUTPUT_METRICS_PATH = Path(r"D:\AI_Bootcamp\genai-smart-education-platform\storage\results\ragas_metrics.xlsx")

TOP_K = 4

In [2]:
import sys
import types
import uuid
from typing import Any, Dict, List

import pandas as pd
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from backend.config import OPENAI_CHAT_MODEL, OPENAI_EMBEDDING_MODEL
from backend.module_1.chat import run_chat_workflow

c:\Users\gg\anaconda3\envs\agenticai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def _llm(temperature: float = 0.0) -> ChatOpenAI:
    return ChatOpenAI(model=OPENAI_CHAT_MODEL, temperature=temperature)


def _embeddings() -> OpenAIEmbeddings:
    return OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)


def _normalise_eval_columns(df: pd.DataFrame) -> pd.DataFrame:
    lower_map = {str(column).strip().lower(): column for column in df.columns}
    question_col = lower_map.get("questions") or lower_map.get("question")
    answer_col = (
        lower_map.get("reference_answers")
        or lower_map.get("reference_answer")
        or lower_map.get("reference")
        or lower_map.get("ground_truth")
        or lower_map.get("ground_truths")
        or lower_map.get("answer")
    )
    category_col = lower_map.get("category")
    if not question_col or not answer_col or not category_col:
        raise ValueError(
            "Excel must contain a question column and a reference answer column. "
            "Accepted reference column names: reference_answer, reference, "
            "ground_truth, ground_truths, answer. A Category column is also required."
        )
    out = df[[question_col, answer_col, category_col]].copy()
    out.columns = ["question", "reference_answer", "category"]
    out = out.dropna(subset=["question", "reference_answer"])
    out["question"] = out["question"].astype(str)
    out["reference_answer"] = out["reference_answer"].astype(str)
    out["category"] = out["category"].fillna("").astype(str).str.strip()
    return out

In [4]:
def _patch_ragas_optional_vertexai_import() -> None:
    """Satisfy an unused optional import in older RAGAS/LangChain combinations."""
    module_name = "langchain_community.chat_models.vertexai"
    if module_name in sys.modules:
        return

    vertexai_stub = types.ModuleType(module_name)

    class ChatVertexAI:
        def __init__(self, *args: Any, **kwargs: Any) -> None:
            raise ImportError(
                "ChatVertexAI stub was called. This evaluation uses OpenAI, "
                "so VertexAI should not be instantiated."
            )

    vertexai_stub.ChatVertexAI = ChatVertexAI
    sys.modules[module_name] = vertexai_stub


async def _score_single_sample(sample: Any, metrics: List[Any]) -> Dict[str, float]:
    metric_scores = {}
    for metric in metrics:
        score = await metric.single_turn_ascore(sample)
        metric_name = getattr(metric, "name", metric.__class__.__name__)
        metric_scores[metric_name] = float(score) if score is not None else None
    return metric_scores


async def _run_ragas_metrics_direct(records: List[Dict[str, Any]]) -> List[Dict[str, float]]:
    _patch_ragas_optional_vertexai_import()

    try:
        from ragas.dataset_schema import SingleTurnSample
        from ragas.embeddings import LangchainEmbeddingsWrapper
        from ragas.llms import LangchainLLMWrapper
        from ragas.metrics import Faithfulness, LLMContextRecall, ResponseRelevancy
        try:
            from ragas.metrics import LLMContextPrecisionWithReference as ContextPrecisionMetric
        except ImportError:
            from ragas.metrics import LLMContextPrecisionWithoutReference as ContextPrecisionMetric
    except Exception as exc:
        raise RuntimeError(
            "Could not import direct RAGAS metric classes. Run: "
            "pip install -U ragas langchain-openai openpyxl pandas. "
            f"Original error: {type(exc).__name__}: {exc}"
        ) from exc

    evaluator_llm = LangchainLLMWrapper(_llm(temperature=0.0))
    evaluator_embeddings = LangchainEmbeddingsWrapper(_embeddings())
    metrics = [
        Faithfulness(llm=evaluator_llm),
        ContextPrecisionMetric(llm=evaluator_llm),
        LLMContextRecall(llm=evaluator_llm),
        ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
    ]

    results = []
    for record in records:
        sample = SingleTurnSample(
            user_input=record["question"],
            response=record["generated_answer"],
            reference=record["reference_answer"],
            retrieved_contexts=record["retrieved_chunks_list"],
        )
        results.append(await _score_single_sample(sample, metrics))
    return results

In [5]:
async def run_ragas_evaluation(
    input_path: Path, output_path: Path, top_k: int = 4
) -> Dict[str, Any]:
    input_path = Path(input_path)
    output_path = Path(output_path)
    if not input_path.exists():
        raise FileNotFoundError(f"Excel file not found: {input_path}")
    if output_path.suffix.lower() != ".xlsx":
        raise ValueError("OUTPUT_METRICS_PATH must end with .xlsx")

    eval_df = _normalise_eval_columns(pd.read_excel(input_path))
    detail_rows = []
    ragas_records = []
    theoretical_row_indices = []

    for _, row in eval_df.iterrows():
        result = run_chat_workflow(
            row["question"], top_k=top_k,
            session_id=f"eval-{uuid.uuid4().hex}", use_memory=False,
        )
        contexts = result["contexts"]
        output_row_index = len(detail_rows)
        detail_rows.append({
            "question": row["question"],
            "redacted_question": result["redacted_question"],
            "pii_redaction_log": "; ".join(result["pii_redaction_log"]),
            "answer_pii_redaction_log": "; ".join(result["answer_pii_redaction_log"]),
            "category": row["category"],
            "workflow_query_type": result["query_type"],
            "reference_answer": row["reference_answer"],
            "generated_answer": result["answer"],
            "retrieved_chunks": "\n\n---CHUNK---\n\n".join(contexts),
            "retrieved_metadata": str(result["metadata"]),
        })
        if row["category"].casefold() == "theoretical":
            theoretical_row_indices.append(output_row_index)
            ragas_records.append({
                "question": result["redacted_question"],
                "reference_answer": row["reference_answer"],
                "generated_answer": result["answer"],
                "retrieved_chunks_list": contexts,
            })

    metric_rows = await _run_ragas_metrics_direct(ragas_records) if ragas_records else []
    final_df = pd.DataFrame(detail_rows)
    metrics_df = pd.DataFrame(metric_rows)
    rename_map = {
        "Faithfulness": "faithfulness",
        "llm_context_precision_with_reference": "context_precision",
        "llm_context_precision_without_reference": "context_precision",
        "LLMContextPrecisionWithReference": "context_precision",
        "LLMContextPrecisionWithoutReference": "context_precision",
        "llm_context_recall": "context_recall",
        "LLMContextRecall": "context_recall",
        "response_relevancy": "answer_relevancy",
        "ResponseRelevancy": "answer_relevancy",
    }
    metrics_df = metrics_df.rename(columns=lambda c: rename_map.get(c, c))
    metric_names = ["faithfulness", "context_precision", "context_recall", "answer_relevancy"]
    for column in metric_names:
        final_df[column] = None

    for metric_index, output_row_index in enumerate(theoretical_row_indices):
        for column in metric_names:
            if column in metrics_df.columns:
                final_df.at[output_row_index, column] = metrics_df.at[metric_index, column]

    summary_rows = []
    for column in metric_names:
        mean_score = pd.to_numeric(final_df[column], errors="coerce").mean()
        summary_rows.append({
            "metric": column,
            "mean_score": float(mean_score) if pd.notna(mean_score) else None,
        })
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        final_df.to_excel(writer, index=False, sheet_name="results")
        pd.DataFrame(summary_rows).to_excel(writer, index=False, sheet_name="summary")

    return {
        "message": "RAGAS evaluation completed.",
        "rows_processed": len(final_df),
        "rows_evaluated": len(theoretical_row_indices),
        "rows_skipped": len(final_df) - len(theoretical_row_indices),
        "summary": summary_rows,
        "result_file": str(output_path.resolve()),
    }

In [6]:
evaluation_result = await run_ragas_evaluation(
    INPUT_DATASET_PATH, OUTPUT_METRICS_PATH, top_k=TOP_K
)
evaluation_result

C:\Users\gg\AppData\Local\Temp\ipykernel_15648\67780857.py:36: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, LLMContextRecall, ResponseRelevancy
C:\Users\gg\AppData\Local\Temp\ipykernel_15648\67780857.py:36: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import Faithfulness, LLMContextRecall, ResponseRelevancy
C:\Users\gg\AppData\Local\Temp\ipykernel_15648\67780857.py:36: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRe

{'message': 'RAGAS evaluation completed.',
 'rows_processed': 89,
 'rows_evaluated': 61,
 'rows_skipped': 28,
 'summary': [{'metric': 'faithfulness', 'mean_score': 0.815387777068105},
  {'metric': 'context_precision', 'mean_score': 0.7600182149007817},
  {'metric': 'context_recall', 'mean_score': 0.6917252146760343},
  {'metric': 'answer_relevancy', 'mean_score': 0.790125504438806}],
 'result_file': 'D:\\AI_Bootcamp\\genai-smart-education-platform\\storage\\results\\ragas_metrics.xlsx'}